In [1]:
# Purpose: Import necessary libraries for data analysis, visualization, and geographic mapping.
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
import numpy as np
import re
from copy import deepcopy
import json
import us

Matplotlib is building the font cache; this may take a moment.


In [ ]:
# Purpose: Define a recursive function to traverse an augur JSON tree, extract state transitions, host information, and identify potential spillover events.
#recurse throught the json file to get edges between states
#create dataframe with columns state1, state2, divergence
hostCount = 0
child_count=0
hostMismatch =[]
nextstrainMismatch = []
spillover_data = {}
squandered_count=0
#hostLookup =bv_brc.set_index('Strain')['Host Name'].to_dict()
#hostLookup =nextstrain.set_index('strain')['host_category'].to_dict()

from datetime import datetime, timedelta

def decimal_date_to_datetime(dec_date):
    """Converts a decimal date to a datetime object."""
    if dec_date is None:
        return None
    year = int(dec_date)
    rem = dec_date - year
    base = datetime(year, 1, 1)
    # Check for leap year to use correct number of days
    days_in_year = 366 if (year % 4 == 0 and year % 100 != 0) or (year % 400 == 0) else 365
    result = base + timedelta(seconds=(rem * days_in_year * 24 * 3600))
    return result

hostLookup ={}
spillover_table = pd.DataFrame(columns=[
    'spillover_type', 'decimal_date', 'state', 'day_interval', 
    'ancestor_day_interval', 'leaf_count', 'spillover_id', 'contributing_nodes'
])
leaf_table= pd.DataFrame(columns=['location', 'host', 'date', 'name'])
def year_month(date):
    year = int(date)
    month = int((date - int(date))*12)
    if month == 0:
        month = 1
    return f"{year}-{month:02d}"

from itertools import combinations

def augur_recurse(node, div_df, parentState=None, parentDiv=None, parentHost=None, init_count=None):
    global child_count, hostCount, hostMismatch, spillover_table, leaf_table, hostLookup, spillover_data, squandered_count

    if init_count != None:
        child_count=init_count
    state = node.get("node_attrs", {}).get("division", {}).get("value", None)
    if state!=None and len(state) == 2:
        state_obj=us.states.lookup(state)
        if state_obj is not None:
            if state_obj.name == "Puerto Rico":
                print(f" hmm{state}")
            state=state_obj.name
    divergence = node.get("node_attrs", {}).get("div", {})
    host = node.get("node_attrs", {}).get("host", {}).get("value", "no host")
    date= node.get("node_attrs", {}).get("num_date", {}).get("value", None)
    name= node.get("name", None)
    node_attrs = node.get("node_attrs", {})
    #state = node_attrs.get("division", {}).get("value", None)
    host = node_attrs.get("host", {}).get("value", "no host")
    date = node_attrs.get("num_date", {}).get("value", None)
    county = node_attrs.get("county", {}).get("value", None)
    
    # list of evidence dictionaries passed up the tree
    evidence_list = []

    if not name.startswith("NODE"):
        child_count += 1
        if name in hostLookup:
            host= hostLookup[name]
            hostCount+=1
        else:
            hostMismatch.append(name)
        
        if host == "Avian":
            if len(name.split("/")) > 2:
                host=categorize_host(name.split("/")[1])
                print(f"Avian {name}")

        if host != "no host":
            evidence_list.append({'name': name, 'host': host, 'state': state, 'date': date, 'county': county})

        # Populate leaf_table
        leaf_table = pd.concat([leaf_table, pd.DataFrame([{
            'location': state, 'host': host, 'date': date, 'name': name, 'county': county
        }])], ignore_index=True)

    #Recursive Step: Aggregate evidence from children
    for child in node.get("children", []):
        div_df, child_evidence_list = augur_recurse(child, div_df, state, date, host)
        evidence_list.extend(child_evidence_list)

    if len(evidence_list) > 1: # Only process if there are at least two leaves to form a pair
        
        # Check for spillover evidence among all pairs of leaves
        for (e1, e2) in combinations(evidence_list, 2):
                
                spillover_type = f"{e1['county']} to {e2['county']}"
                spillover_id = f"{e1['name']}->{e2['name']}"
                spillover_entry = {
                    'spillover_type': spillover_type,
                    'decimal_date': date,
                    'state': state,
                    'day_interval': None,  # Placeholder, can be calculated if needed
                    'ancestor_day_interval': None,  # Placeholder, can be calculated if needed
                    'leaf_count': len(evidence_list),
                    'spillover_id': spillover_id,
                    'contributing_nodes': [e1['name'], e2['name']]
                }
                spillover_table = pd.concat([spillover_table, pd.DataFrame([spillover_entry])], ignore_index=True)
    # If no spillovers were found, return all evidence
    return div_df, evidence_list


In [7]:
div_df = pd.DataFrame(columns=['state1', 'state2', 'divergence'])
spillover_table = pd.DataFrame(columns=['spillover_type', 'decimal_date', 'state', 'all_host', 'day_interval', 'spillover_id', 'ancestor_day_interval', 'leaf_count'])
leaf_table= pd.DataFrame(columns=['location', 'host', 'date', 'name'])

with open("../../data/first16k/ncov_with_hcov19_prefix.json") as f:

    augur_output = json.load(f)
    child_count = 0
    div_df, child_data = augur_recurse(list(augur_output.items())[2][1], div_df, init_count=0)
    print(f"child count is {child_count}")

/var/folders/pq/2t2l79812g3fsrtcvgbf15580000gq/T/ipykernel_13916/2056448798.py:83: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  leaf_table = pd.concat([leaf_table, pd.DataFrame([{


child count is 16522


In [8]:
leaf_table.head()

,location,host,date,name
0,None,no host,2019.985,Wuhan-Hu-1/2019
1,None,no host,2021.410,USA/VA-CDC-LC0230473/2021
2,None,no host,2021.426,USA/VA-EHip-366271206.187/2021
3,None,no host,2021.440,USA/VA-EHip-366271208.192/2021
4,None,no host,2021.440,USA/VA-EHip-367438056.192/2021
